In [ ]:
!pip install torch-geometric -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# **Attention in GAT**

Working mechanism:
GCN aggregation for node v:
 h_v = W · SUM( (1/sqrt(d_u * d_v)) * h_u )
 — weight is fixed, based only on node degrees
#
GAT aggregation for node v:
 h_v = W · SUM( alpha_uv * h_u )
 — weight alpha_uv is LEARNED from features

How alpha_uv is computed:
e_uv   = LeakyReLU( a^T · [W*h_u || W*h_v] )  ← raw score
 alpha_uv = softmax over neighbors of e_uv        ← normalized
#
Multi-head attention (heads=8):
Run 8 independent attention mechanisms in parallel
Concatenate their outputs → richer representation

# **Loading both Cora and Citeseer datasets**

In [ ]:
cora     = Planetoid(root='/tmp/Cora', name='Cora')
data_cora = cora[0].to(device)

citeseer     = Planetoid(root='/tmp/Citeseer', name='Citeseer')
data_citeseer = citeseer[0].to(device)

print("Cora     :", data_cora)
print("Citeseer :", data_citeseer)

Processing...
Done!
Processing...


Cora     : Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
Citeseer : Data(x=[3327, 3703], edge_index=[2, 9104], y=[3327], train_mask=[3327], val_mask=[3327], test_mask=[3327])


Done!


# **GAT model**

In [ ]:

class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels,
                 out_channels, heads=8):
        super().__init__()

        # Layer 1: multi-head attention
        # heads=8 runs 8 attention mechanisms in parallel
        # output size = hidden_channels * heads
        self.conv1 = GATConv(in_channels, hidden_channels,
                              heads=heads, dropout=0.6)

        # Layer 2: single-head attention (concat=False → averages heads)
        # output size = out_channels
        self.conv2 = GATConv(hidden_channels * heads, out_channels,
                              heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)               # ELU instead of ReLU (from paper)
        x = F.dropout(x, p=0.6, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def forward_with_attention(self, x, edge_index):
        x = F.dropout(x, p=0.6, training=self.training)
        x, attn1 = self.conv1(x, edge_index,
                              return_attention_weights=True)
        x = F.elu(x)
        x = F.dropout(x, p=0.6, training=self.training)
        x, attn2 = self.conv2(x, edge_index,
                              return_attention_weights=True)
        return x, attn1, attn2

def make_gat(dataset):
    return GAT(
        in_channels     = dataset.num_node_features,
        hidden_channels = 8,    # 8 units × 8 heads = 64 in layer 1
        out_channels    = dataset.num_classes,
        heads           = 8
    ).to(device)

model_cora = make_gat(cora)
print(model_cora)
print(f"\nParameters: {sum(p.numel() for p in model_cora.parameters()):,}")

GAT(
  (conv1): GATConv(1433, 8, heads=8)
  (conv2): GATConv(64, 7, heads=1)
)

Parameters: 92,373


# **Before training snapshot**

In [ ]:
model_cora.eval()
with torch.no_grad():
    out_before  = model_cora(data_cora.x, data_cora.edge_index)
    pred_before = out_before.argmax(dim=1)

acc_before = (pred_before[data_cora.test_mask] ==
              data_cora.y[data_cora.test_mask])
acc_before = acc_before.sum().item() / data_cora.test_mask.sum().item()
print(f"Accuracy BEFORE training : {acc_before*100:.1f}%")
print(f"Random chance baseline   : {100/7:.1f}%")

test_indices = data_cora.test_mask.nonzero(as_tuple=True)[0][:10]

Accuracy BEFORE training : 13.9%
Random chance baseline   : 14.3%


##  Train on Cora

In [ ]:

def train(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask],
                             data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out  = model(data.x, data.edge_index)
        pred = out.argmax(dim=1)
        acc  = (pred[mask] == data.y[mask]).sum() / mask.sum()
    return acc.item()


optimizer    = torch.optim.Adam(model_cora.parameters(),
                                lr=0.005, weight_decay=5e-4)
train_losses = []
val_accs     = []

for epoch in range(1, 201):
    loss    = train(model_cora, data_cora, optimizer)
    val_acc = evaluate(model_cora, data_cora, data_cora.val_mask)
    train_losses.append(loss)
    val_accs.append(val_acc)
    if epoch % 20 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch 020 | Loss: 0.9077 | Val Acc: 0.7560
Epoch 040 | Loss: 0.6708 | Val Acc: 0.7780
Epoch 060 | Loss: 0.5938 | Val Acc: 0.7720
Epoch 080 | Loss: 0.4737 | Val Acc: 0.7660
Epoch 100 | Loss: 0.4983 | Val Acc: 0.7720
Epoch 120 | Loss: 0.5756 | Val Acc: 0.7700
Epoch 140 | Loss: 0.4102 | Val Acc: 0.7820
Epoch 160 | Loss: 0.3729 | Val Acc: 0.7760
Epoch 180 | Loss: 0.3429 | Val Acc: 0.7740
Epoch 200 | Loss: 0.4804 | Val Acc: 0.7940


# **Visualize attention weights on node 0**

In [ ]:
# After training, inspecting what node 0 attends to
# attention weights tell us which neighbors matter most

model_cora.eval()
with torch.no_grad():
    _, attn1, _ = model_cora.forward_with_attention(
        data_cora.x, data_cora.edge_index
    )

edge_index_attn, weights = attn1

# weights shape: [num_edges, num_heads]
# average across 8 heads for visualization
avg_weights = weights.mean(dim=1).cpu().numpy()

# Finding all edges where node 0 is the destination

dst_nodes = edge_index_attn[1].cpu().numpy()
src_nodes = edge_index_attn[0].cpu().numpy()
mask_node0 = dst_nodes == 0

print("Attention weights for node 0's neighbors:")
print(f"{'Neighbor':<12} {'Attention weight':>18} {'Bar'}")
print("-" * 50)
for src, w in zip(src_nodes[mask_node0], avg_weights[mask_node0]):
    bar = "█" * int(w * 200)
    print(f"Node {src:<7} {w:>18.4f}  {bar}")
print(f"\nWeights sum to ~1.0: {avg_weights[mask_node0].sum():.4f}")
print("(softmax normalizes across neighbors)")

Attention weights for node 0's neighbors:
Neighbor       Attention weight Bar
--------------------------------------------------
Node 633                 0.2645  ████████████████████████████████████████████████████
Node 1862                0.2468  █████████████████████████████████████████████████
Node 2582                0.2374  ███████████████████████████████████████████████
Node 0                   0.2513  ██████████████████████████████████████████████████

Weights sum to ~1.0: 1.0000
(softmax normalizes across neighbors)


# **Train and test on Citeseer**

In [ ]:
model_citeseer = make_gat(citeseer)
optimizer_cs   = torch.optim.Adam(model_citeseer.parameters(),
                                   lr=0.005, weight_decay=5e-4)
val_accs_cs    = []

for epoch in range(1, 201):
    loss    = train(model_citeseer, data_citeseer, optimizer_cs)
    val_acc = evaluate(model_citeseer, data_citeseer,
                        data_citeseer.val_mask)
    val_accs_cs.append(val_acc)
    if epoch % 40 == 0:
        print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | Val Acc: {val_acc:.4f}")

test_cora = evaluate(model_cora, data_cora, data_cora.test_mask)
test_cs   = evaluate(model_citeseer, data_citeseer,
                      data_citeseer.test_mask)

print(f"\nCora  test accuracy : {test_cora*100:.1f}%")
print(f"Citeseer test accuracy : {test_cs*100:.1f}%")

Epoch 040 | Loss: 0.5615 | Val Acc: 0.6820
Epoch 080 | Loss: 0.5358 | Val Acc: 0.6620
Epoch 120 | Loss: 0.5061 | Val Acc: 0.6640
Epoch 160 | Loss: 0.4439 | Val Acc: 0.6760
Epoch 200 | Loss: 0.5290 | Val Acc: 0.6720

Cora  test accuracy : 81.1%
Citeseer test accuracy : 65.8%


# **Improvement stage of the current GAT framework**

--> Changing dropout rate

--> Altering learning rate to 0.01 instead of the previous 0.005

--> Adding early stopping

In [ ]:
# Improved GAT with tuned hyperparameters
class GATImproved(nn.Module):
    def __init__(self, in_channels, hidden_channels,
                 out_channels, heads=8, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GATConv(in_channels, hidden_channels,
                             heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_channels * heads, out_channels,
                             heads=1, concat=False, dropout=dropout)

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

In [ ]:
def train_with_early_stopping(model, data, epochs=500,
                               lr=0.01, weight_decay=5e-4,
                               patience=50):
    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=lr,
                                  weight_decay=weight_decay)

    best_val_acc  = 0
    best_weights  = None
    patience_count = 0
    train_losses  = []
    val_accs      = []

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        optimizer.zero_grad()
        out  = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask],
                               data.y[data.train_mask])
        loss.backward()
        optimizer.step()

        # Evaluate
        model.eval()
        with torch.no_grad():
            out     = model(data.x, data.edge_index)
            pred    = out.argmax(dim=1)
            val_acc = (pred[data.val_mask] == data.y[data.val_mask]
                      ).sum().item() / data.val_mask.sum().item()

        train_losses.append(loss.item())
        val_accs.append(val_acc)

        # Early stopping — save best weights
        if val_acc > best_val_acc:
            best_val_acc   = val_acc
            best_weights   = {k: v.clone()
                              for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        if epoch % 50 == 0:
            print(f"Epoch {epoch:03d} | Loss: {loss:.4f} | "
                  f"Val Acc: {val_acc:.4f} | "
                  f"Best: {best_val_acc:.4f} | "
                  f"Patience: {patience_count}/{patience}")

        if patience_count >= patience:
            print(f"\nEarly stopping at epoch {epoch}")
            print(f"Best val acc: {best_val_acc:.4f}")
            break

    # Restore best weights before returning
    model.load_state_dict(best_weights)
    return train_losses, val_accs

## **Trying different dropout rate values on citeseer and picking the best one**

In [ ]:

print("=== Tuning dropout on Citeseer ===\n")

results = {}
for dropout in [0.3, 0.4, 0.5, 0.6]:
    model_tuned = GATImproved(
        in_channels     = citeseer.num_node_features,
        hidden_channels = 8,
        out_channels    = citeseer.num_classes,
        heads           = 8,
        dropout         = dropout
    ).to(device)

    print(f"--- Dropout: {dropout} ---")
    train_with_early_stopping(
        model_tuned, data_citeseer,
        epochs=500, lr=0.01,
        weight_decay=5e-4, patience=50
    )

    test_acc = evaluate(model_tuned, data_citeseer,
                        data_citeseer.test_mask)
    results[dropout] = test_acc
    print(f"Test accuracy: {test_acc*100:.1f}%\n")

print("=== Summary ===")
for dropout, acc in results.items():
    bar = "█" * int(acc * 50)
    print(f"Dropout {dropout}: {acc*100:.1f}%  {bar}")

best_dropout = max(results, key=results.get)
print(f"\nBest dropout: {best_dropout} → {results[best_dropout]*100:.1f}%")

=== Tuning dropout on Citeseer ===

--- Dropout: 0.3 ---
Epoch 050 | Loss: 0.1461 | Val Acc: 0.6820 | Best: 0.7060 | Patience: 47/50

Early stopping at epoch 53
Best val acc: 0.7060
Test accuracy: 69.7%

--- Dropout: 0.4 ---
Epoch 050 | Loss: 0.1927 | Val Acc: 0.6720 | Best: 0.6960 | Patience: 46/50

Early stopping at epoch 54
Best val acc: 0.6960
Test accuracy: 68.5%

--- Dropout: 0.5 ---
Epoch 050 | Loss: 0.2602 | Val Acc: 0.6540 | Best: 0.6760 | Patience: 17/50
Epoch 100 | Loss: 0.2639 | Val Acc: 0.6440 | Best: 0.6800 | Patience: 19/50

Early stopping at epoch 131
Best val acc: 0.6800
Test accuracy: 68.5%

--- Dropout: 0.6 ---
Epoch 050 | Loss: 0.5877 | Val Acc: 0.6740 | Best: 0.6960 | Patience: 10/50

Early stopping at epoch 90
Best val acc: 0.6960
Test accuracy: 69.3%

=== Summary ===
Dropout 0.3: 69.7%  ██████████████████████████████████
Dropout 0.4: 68.5%  ██████████████████████████████████
Dropout 0.5: 68.5%  ██████████████████████████████████
Dropout 0.6: 69.3%  ██████████████

# **Final Comparison**

In [ ]:
print("======= GAT Citeseer: original vs improved =======")
print(f"Original  (dropout=0.6, lr=0.005, 200 epochs) : 65.8%")
print(f"Improved  (dropout={best_dropout}, "
      f"lr=0.01, early stopping)  : "
      f"{results[best_dropout]*100:.1f}%")
print(f"Gain: +{results[best_dropout]*100 - 65.8:.1f}%")

======= GAT Citeseer: original vs improved =======
Original  (dropout=0.6, lr=0.005, 200 epochs) : 65.8%
Improved  (dropout=0.3, lr=0.01, early stopping)  : 69.7%
Gain: +3.9%
